# Notebook 01 — SAE Setup and Verification
**Owner: Person A — Week 1**

Stage 1 of the pipeline. Load DINOv2 via Prisma, load the pre-trained SAE for layer 11, and verify reconstruction quality and L0 sparsity.

All logic lives in `src/`. This notebook only calls functions.

## 1. Config and imports

In [ ]:
from src.config import get_config
cfg = get_config()

print(cfg.model.name)
print("Primary layer:", cfg.sae.primary_layer)

## 2. Load model

Call `model.get_model()`. Should print model name, device, and parameter count.

In [ ]:
from src.model import get_model
model = get_model()
# Confirm output above shows correct model and device

## 3. Load SAE

Call `sae.get_sae()` for the primary layer. Print dictionary size.

In [ ]:

# Requires SAE weights at outputs/saes/layer_{N}/sae_weights.pt
# Skip until weights are downloaded.
from src.sae import get_sae, encode, decode, ablate_feature

sae = get_sae()                          # loads cfg.sae.primary_layer (11)
print(f"d_in={sae.d_in}, d_sae={sae.d_sae}")
print(f"W_enc: {sae.W_enc.shape}")       # (d_model, d_sae)
print(f"W_dec: {sae.W_dec.shape}")       # (d_sae, d_model)

# encode/decode round-trip
features = encode(activations)           # (1, 197, d_sae)
reconstructed = decode(features)         # (1, 197, 768)
print(f"features shape:      {features.shape}")
print(f"reconstructed shape: {reconstructed.shape}")

# ablate_feature: zero one feature, check target is 0, others unchanged
ablated = ablate_feature(activations, feature_idx=0)
features_after = encode(ablated)
assert features_after[..., 0].abs().max().item() < 1e-6, "Feature 0 not zeroed"
assert (features_after[..., 1:] - features[..., 1:]).abs().max().item() < 1e-6, \
    "Other features changed unexpectedly"
print("ablate_feature OK")


## 4. Single-image smoke test

Load one ImageNet image, run `model.run_with_cache()`, extract the residual stream at the primary layer.

In [ ]:

# Smoke test: synthetic input (no real image needed yet)
import torch
from src.model import get_model
from src.config import get_config

cfg = get_config()
model = get_model()

pixels = torch.randn(1, 3, cfg.model.image_size, cfg.model.image_size)
device = next(model.parameters()).device
pixels = pixels.to(device)

logits, act_cache = model.run_with_cache(pixels)

# Confirm hook key format — share with Person C before Day 4
resid_keys = [k for k in act_cache.keys() if "resid" in k]
print("Residual stream keys:", resid_keys)

# Extract primary layer activations
hook_key = f"blocks.{cfg.sae.primary_layer}.hook_resid_post"
assert hook_key in act_cache, f"Expected key '{hook_key}' not found. Update hook_key above."
activations = act_cache[hook_key]
print(f"activations.shape: {activations.shape}")  # expected (1, seq_len, d_model)

# Cache test: second call must return the same object
model2 = get_model()
assert model is model2, "get_model() returned a different object — caching is broken"
print("Cache check passed.")


In [ ]:
# Download first: python data/load_data.py
import torch
from pathlib import Path
from src.model import preprocess_image
import src.config as _cfg_mod

# Anchor to repo root so this works regardless of notebook cwd
repo_root = _cfg_mod._DEFAULT_CONFIG.parent.parent
image_dir = repo_root / cfg.data.imagenet_val_path

test_image = next(image_dir.glob("**/*.JPEG"), None)

if test_image:
    tensor = preprocess_image(str(test_image))
    expected = (1, 3, cfg.model.image_size, cfg.model.image_size)
    assert tensor.shape == torch.Size(expected), f"Got {tensor.shape}, expected {expected}"
    print(f"preprocess_image OK — shape {tuple(tensor.shape)}, file: {test_image.name}")
else:
    print(f"No images found in {image_dir}. Run: python data/load_data.py")# Download first: python data/load_data.py

## 5. L0 sparsity

Target: < `cfg.sae.l0_target`

In [ ]:

from src.sae import get_l0_sparsity

l0 = get_l0_sparsity(activations)
print(f"L0: {l0:.1f}  (target < {cfg.sae.l0_target})")
assert l0 < cfg.sae.l0_target, f"L0 {l0:.1f} exceeds target {cfg.sae.l0_target}"


## 6. Reconstruction loss

Target: < 0.05 (5%)

In [ ]:

from src.sae import get_reconstruction_loss

loss = get_reconstruction_loss(activations)
print(f"Reconstruction loss: {loss:.4f}  (target < 0.05)")
assert loss < 0.05, f"Reconstruction loss {loss:.4f} exceeds 5% target"


## 7. Dead feature fraction

Requires `evaluate.compute_dead_feature_fraction()` — coordinate with Person C.

## 8. Hook intervention smoke test

Verify `model.run_with_hooks()` — needed before implementing `causal.py`.

Patches layer 11's residual stream with zeros and confirms logits change. Uses the synthetic `pixels` tensor from section 4.

In [ ]:
import torch
from src.model import get_model
from src.config import get_config

cfg = get_config()
model = get_model()

# pixels must already exist from cell c04 (synthetic smoke test)
# If running this cell standalone, re-run cell c04 first.
assert 'pixels' in dir() or 'pixels' in locals(), \
    "Run the smoke test cell (section 4) first to create `pixels`."

layer = cfg.sae.primary_layer  # 11
hook_key = f"blocks.{layer}.hook_resid_post"

# Baseline: normal forward pass
logits_clean, _ = model.run_with_cache(pixels)

# Intervention: zero the residual stream at layer 11
def zero_hook(value, hook):
    return torch.zeros_like(value)

logits_patched = model.run_with_hooks(
    pixels,
    fwd_hooks=[(hook_key, zero_hook)]
)

assert not torch.allclose(logits_clean, logits_patched, atol=1e-4), \
    "Hook had no effect — run_with_hooks intervention is broken"

max_delta = (logits_clean - logits_patched).abs().max().item()
print(f"run_with_hooks OK — max logit delta after zeroing layer {layer}: {max_delta:.4f}")
print(f"  clean   logits[:5]: {logits_clean[0, :5].tolist()}")
print(f"  patched logits[:5]: {logits_patched[0, :5].tolist()}")

# Confirm hooks don't persist across calls
logits_after, _ = model.run_with_cache(pixels)
assert torch.allclose(logits_clean, logits_after, atol=1e-6), \
    "Hooks persisted after run_with_hooks — model state is dirty"
print("Hook isolation OK — clean logits unchanged after patched run.")

In [ ]:
# from src.evaluate import compute_dead_feature_fraction
# dead_frac = compute_dead_feature_fraction(cfg.sae.primary_layer, activations)
# print(f"Dead features: {dead_frac:.1%}")

## Checkpoint
- [ ] Model loads cleanly
- [ ] Hook key format confirmed and shared with Person C
- [ ] L0 < cfg.sae.l0_target
- [ ] Reconstruction loss < 5%
- [ ] Dead feature fraction < 1%
- [ ] `run_with_hooks` intervention verified (section 8)